# 📘 Notebook 3 - Processos ETL nas camadas Bronze, Silver e Gold


---

Nesse notebook teremos a criação das camadas Bronze, Silver e Gold.

Na primeira, armazenaremos em tabelas de banco de dados os dados exatamente como vieram dos arquivos CSV.

Na segunda, faremos limpeza de dados inconsistentes ou inválidos, exclusão de dados desnecessários e a modelagem em **Esquema Estrela**, típico de Data Warehouses.

Por fim, na camada Gold, carregaremos uma tabela _flat_ a partir de junções das tabelas de dados da Silver, de modo que os dados fiquem facilmente acessíveis para as consultas pelo público de interesse (processos de analytics e BI), bem como possam responder às questões previstas nesse MVP.

## 🥉 3.1. Camada Bronze

In [0]:
%sql
-- Criação da base de dados para representar a camada Bronze
CREATE DATABASE IF NOT EXISTS bronze;

In [0]:
# Leitura dos arquivos CSV contidos no DBFS em dataframes PySpark e posterior carga em tabelas da base de dados Bronze

from pyspark.sql import SparkSession
base_path = "/tmp/zip_data"

spark = SparkSession.builder.appName("LerCSV").getOrCreate()

tabelas = ["patients", "organizations", "payers", "encounters", "procedures"]
df = {}
for tabela in tabelas:

    df[tabela] = spark.read.option("header", "true").csv(f"dbfs:{base_path}/{tabela}.csv") 
    df[tabela].write.mode("overwrite").saveAsTable(f"bronze.{tabela}")



In [0]:
#df_patients.limit(10).display()

Tabelas criadas - serão feitas consultas para a visualização dos primeiros registros de cada uma delas.

In [0]:
%sql
SELECT * FROM bronze.patients LIMIT 20

Id,BIRTHDATE,DEATHDATE,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON
5605b66b-e92d-c16c-1b83-b8bf7040d51f,1977-03-19,null,Mrs.,Nikita578,Erdman779,null,Leannon79,M,white,nonhispanic,F,Wakefield Massachusetts US,510 Little Station Unit 69,Quincy,Massachusetts,Norfolk County,02186,42.290937381211286,-70.97550306
6e5ae27c-8038-7988-e2c0-25a103f01bfa,1940-02-19,null,Mr.,Zane918,Hodkiewicz467,null,null,M,white,nonhispanic,M,Brookline Massachusetts US,747 Conn Throughway,Boston,Massachusetts,Suffolk County,02135,42.308831197562505,-71.0631616
8123d076-0886-9007-e956-d5864aa121a7,1958-06-04,null,Mr.,Quinn173,Marquardt819,null,null,M,white,nonhispanic,M,Gardner Massachusetts US,816 Okuneva Extension Apt 91,Quincy,Massachusetts,Norfolk County,02170,42.26517684888508,-70.96708508
770518e4-6133-648e-60c9-071eb2f0e2ce,1928-12-25,2017-09-29,Mr.,Abel832,Smitham825,null,null,M,white,hispanic,M,Randolph Massachusetts US,127 Cole Way Unit 95,Boston,Massachusetts,Suffolk County,02118,42.334303740740594,-71.0668012
f96addf5-81b9-0aab-7855-d208d3d352c5,1928-12-25,2014-02-23,Mr.,Edwin773,Labadie908,null,null,M,white,hispanic,M,Stow Massachusetts US,976 Ziemann Gateway,Boston,Massachusetts,Suffolk County,02125,42.346771403899275,-71.05881297
8e9650d1-788a-78f9-4a28-d08f7f95354a,1928-12-25,null,Mr.,Frankie174,Oberbrunner298,null,null,M,white,hispanic,M,Boston Massachusetts US,303 Bechtelar Bypass Suite 26,Boston,Massachusetts,Suffolk County,02467,42.37102647,-71.11810672
183df435-4190-060e-8f8e-bf63c572b266,1957-11-08,null,Mrs.,Eilene124,Walsh511,null,Wiegand701,M,asian,nonhispanic,F,Beijing Beijing Municipality CN,235 Lang Parade,Cambridge,Massachusetts,Middlesex County,02142,42.35892760552785,-71.15622361
720560d4-51da-c38c-ee90-c15935278df1,1972-06-27,null,Mr.,Lowell343,Price929,null,null,M,white,nonhispanic,M,Lowell Massachusetts US,694 Kuhlman Corner Apt 74,Quincy,Massachusetts,Norfolk County,02170,42.297903944576696,-71.01598304
217851b0-5f47-d376-18b9-0fe4ba77207e,1954-03-06,null,Mr.,Adrian111,Gleason633,null,null,S,black,hispanic,M,Boston Massachusetts US,808 Gottlieb Wall,Boston,Massachusetts,Suffolk County,02126,42.38408414202098,-71.1006892
ff331e5c-ab16-e218-f39a-63e11de1ed75,1927-07-10,null,Mr.,Eugene421,Abernathy524,null,null,M,native,hispanic,M,Pembroke Massachusetts US,706 Connelly Track Unit 1,Boston,Massachusetts,Suffolk County,02111,42.358519313454885,-71.07859759


In [0]:
%sql
SELECT * FROM bronze.organizations 

Id,NAME,ADDRESS,CITY,STATE,ZIP,LAT,LON
d78e84ec-30aa-3bba-a33a-f29a3a454662,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,02114,42.362813,-71.069187


Em se tratando dos dados exclusivos do Hospital Geral de Massachussets, essa tabela só contém uma linha.
De toda forma, o banco de dados fica preparado caso futuramente se deseje agregar dados de outras instituições de saúde.

In [0]:
%sql
SELECT * FROM bronze.payers

Id,NAME,ADDRESS,CITY,STATE_HEADQUARTERED,ZIP,PHONE
b3221cfc-24fb-339e-823d-bc4136cbc4ed,Dual Eligible,7500 Security Blvd,Baltimore,MD,21244,1-877-267-2323
7caa7254-5050-3b5e-9eae-bd5ea30e809c,Medicare,7500 Security Blvd,Baltimore,MD,21244,1-800-633-4227
7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,Medicaid,7500 Security Blvd,Baltimore,MD,21244,1-877-267-2323
d47b3510-2895-3b70-9897-342d681c769d,Humana,500 West Main St,Louisville,KY,40018,1-844-330-7799
6e2f1a2d-27bd-3701-8d08-dae202c58632,Blue Cross Blue Shield,Michigan Plaza,Chicago,IL,60007,1-800-262-2583
5059a55e-5d6e-34d1-b6cb-d83d16e57bcf,UnitedHealthcare,9800 Healthcare Lane,Minnetonka,MN,55436,1-888-545-5205
4d71f845-a6a9-3c39-b242-14d25ef86a8d,Aetna,151 Farmington Ave,Hartford,CT,6156,1-800-872-3862
047f6ec3-6215-35eb-9608-f9dda363a44c,Cigna Health,900 Cottage Grove Rd,Bloomfield,CT,6002,1-800-997-1654
42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,Anthem,220 Virginia Ave,Indianapolis,IN,46204,1-800-331-1476
b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,NO_INSURANCE,null,null,null,null,null


Nessa tabela, o último registro denominado "NO_INSURANCE" se presta a indicar atendimentos hospitalares não cobertos por companhia seguradora. É necessário uma vez que, pelo dicionário de dados que vimos na seção anterior, há um relacionamento entre os dados da tabela _encounters_ com _payers_.

In [0]:
%sql
SELECT * FROM bronze.encounters LIMIT 20

Id,START,STOP,PATIENT,ORGANIZATION,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
32c84703-2481-49cd-d571-3899d5820253,2011-01-02T09:26:36Z,2011-01-02T12:58:36Z,3de74169-7f67-9304-91d4-757e0f3a14d2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,185347001,Encounter for problem (procedure),85.55,1018.02,0,null,null
c98059da-320a-c0a6-fced-c8815f3e3f39,2011-01-03T05:44:39Z,2011-01-03T06:01:42Z,d9ec2e44-32e9-9148-179a-1653348cc4e2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,outpatient,308335008,Patient encounter procedure,142.58,2619.36,0,null,null
4ad28a3a-2479-782b-f29c-d5b3f41a001e,2011-01-03T14:32:11Z,2011-01-03T14:47:11Z,73babadf-5b2b-fee7-189e-6f41ff213e01,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185349003,Encounter for check up (procedure),85.55,461.59,305.27,null,null
c3f4da61-e4b4-21d5-587a-fbc89943bc19,2011-01-03T16:24:45Z,2011-01-03T16:39:45Z,3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,wellness,162673000,General examination of patient (procedure),136.8,1784.24,0,null,null
a9183b4f-2572-72ea-54c2-b3cd038b4be7,2011-01-03T17:36:53Z,2011-01-03T17:51:53Z,fa006887-d93c-d302-8b89-f3c25f88c0e1,d78e84ec-30aa-3bba-a33a-f29a3a454662,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,ambulatory,390906007,Follow-up encounter,85.55,234.72,0,55822004,Hyperlipidemia
c4923a74-3e40-8b0c-cf73-05b9c0390621,2011-01-03T19:08:16Z,2011-01-03T19:23:16Z,823c6b40-9dbe-e463-310b-ea2b23b23b48,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,wellness,162673000,General examination of patient (procedure),136.8,1183.25,946.58,null,null
c140ed81-040e-8319-e860-f72b4738ed22,2011-01-03T22:39:50Z,2011-01-03T22:54:50Z,9c616fc0-00ea-249f-d073-1f3bb15d41fa,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,outpatient,185349003,Encounter for check up (procedure),85.55,6024.77,0,null,null
2cfd4ddd-ad13-fe1e-528b-15051cea2ec3,2011-01-04T14:49:55Z,2011-01-04T15:04:55Z,d856d6e6-4c98-e7a2-129b-44076c63d008,d78e84ec-30aa-3bba-a33a-f29a3a454662,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,ambulatory,185347001,Encounter for problem,85.55,11855.19,11205.43,363406005,Malignant tumor of colon
16bdc066-886f-34e1-38fa-afb85090b637,2011-01-04T15:13:10Z,2011-01-04T15:28:10Z,a80b1160-93f0-db7e-9f23-04ea6fdddfaf,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,wellness,162673000,General examination of patient (procedure),136.8,272.8,0,null,null
17966936-0878-f4db-128b-a43ae10d0878,2011-01-05T04:02:09Z,2011-01-05T04:17:09Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185347001,Encounter for problem,85.55,9881.17,7872.94,254637007,Non-small cell lung cancer (disorder)


In [0]:
%sql
SELECT * FROM bronze.procedures LIMIT 20

START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,REASONCODE,REASONDESCRIPTION
2011-01-02T09:26:36Z,2011-01-02T12:58:36Z,3de74169-7f67-9304-91d4-757e0f3a14d2,32c84703-2481-49cd-d571-3899d5820253,265764009,Renal dialysis (procedure),903,null,null
2011-01-03T05:44:39Z,2011-01-03T06:01:42Z,d9ec2e44-32e9-9148-179a-1653348cc4e2,c98059da-320a-c0a6-fced-c8815f3e3f39,76601001,Intramuscular injection,2477,null,null
2011-01-04T14:49:55Z,2011-01-04T15:04:55Z,d856d6e6-4c98-e7a2-129b-44076c63d008,2cfd4ddd-ad13-fe1e-528b-15051cea2ec3,703423002,Combined chemotherapy and radiation therapy (procedure),11620,363406005,Malignant tumor of colon
2011-01-05T04:02:09Z,2011-01-05T04:17:09Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,17966936-0878-f4db-128b-a43ae10d0878,173160006,Diagnostic fiberoptic bronchoscopy (procedure),9796,162573006,Suspected lung cancer (situation)
2011-01-05T12:58:36Z,2011-01-05T16:42:36Z,3de74169-7f67-9304-91d4-757e0f3a14d2,9de5f0b0-4ba4-ce6f-45fb-b55c202f31a5,265764009,Renal dialysis (procedure),1255,null,null
2011-01-06T18:12:41Z,2011-01-06T18:17:12Z,712e95dd-f313-39c7-1fc9-8463c15c105f,44057ea7-178f-c3c3-965a-78e4521fa43b,410006001,Digital examination of rectum,564,null,null
2011-01-07T21:02:09Z,2011-01-07T22:39:45Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,4b6a38e6-7df9-2d75-d850-6bd662f3d05b,698354004,Magnetic resonance imaging for measurement of brain volume (procedure),5050,424132000,Non-small cell carcinoma of lung TNM stage 1 (disorder)
2011-01-07T22:39:45Z,2011-01-07T23:00:45Z,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,cc36dc26-6019-94a5-93b7-7bbf312e6fc4,703423002,Combined chemotherapy and radiation therapy (procedure),15760,424132000,Non-small cell carcinoma of lung TNM stage 1 (disorder)
2011-01-08T16:42:36Z,2011-01-08T20:15:36Z,3de74169-7f67-9304-91d4-757e0f3a14d2,03f54837-bfc8-81aa-4905-f74ceb35162f,265764009,Renal dialysis (procedure),1556,null,null
2011-01-09T16:11:28Z,2011-01-09T16:51:19Z,f4cde0dc-db22-5d98-5228-55121ae74cad,17d162fa-fc25-b3c5-5e72-20287a55eaf1,73761001,Colonoscopy,13620,null,null


## 🥈 3.2. Camada Silver

Na construção dessa camada, é proposto um **Esquema Estrela**, composto por uma tabela **Fato** e três tabelas **Dimensão**. 

Todas são derivadas das tabelas da camada Bronze, após algumas limpezas. Também percebe-se que foi optado por não trazer para a Silver a tabela _Procedures_ da camada anterior, visto que os dados ali presentes não são necessários para responder às perguntas do problema e esta tabela possui muitos nulos, como vimos na etapa de Qualidade dos Dados.

A modelagem proposta pode ser vista na figura abaixo:

![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/diagrama_silver.png)

A seguir, temos o código SQL de criação das tabelas da camada Silver.

In [0]:
%sql
-- Criação da base de dados para representar a camada Silver
CREATE DATABASE IF NOT EXISTS silver;

### 🧾 Dimensão  Pacientes

Na primeira observação que fizemos em amostra dos dados dos pacientes, constatamos de que as partes que compõem os nomes (First, Last, Maiden) contêm caraceteres numéricos, por motivos que desconhecemos.

In [0]:
%sql

-- Visualização dos nomes contendo caracteres numéricos

SELECT First, Last, Maiden FROM bronze.patients LIMIT 20

First,Last,Maiden
Nikita578,Erdman779,Leannon79
Zane918,Hodkiewicz467,null
Quinn173,Marquardt819,null
Abel832,Smitham825,null
Edwin773,Labadie908,null
Frankie174,Oberbrunner298,null
Eilene124,Walsh511,Wiegand701
Lowell343,Price929,null
Adrian111,Gleason633,null
Eugene421,Abernathy524,null


Também constatamos caracteres inválidos no nome dos pacientes. 

In [0]:
%sql

-- Busca de pacientes cujo nome contenha caracteres inválidos, isto é, que não sejam letras, números, espaço, apóstrofo ou underscore

SELECT First, Last, Maiden
FROM bronze.patients
WHERE First RLIKE concat('[^a-zA-Z0-9_ ', chr(39), ']')
   OR Last RLIKE concat('[^a-zA-Z0-9_ ', chr(39), ']')
   OR Maiden RLIKE concat('[^a-zA-Z0-9_ ', chr(39), ']')

First,Last,Maiden
Ad√°n600,Feliciano160,null
Lucas404,Su√°rez24,null
Jos√© Mar√≠a211,Delgadillo349,null
Jos√© Emilio366,Santill√°n790,null
Bernardo699,Fl√≥rez858,null
Carlos172,Far√≠as160,null
Daniela614,Ochoa950,Avil√©s474
Jos√© Eduardo181,Viera553,null
Nicol√°s801,Miranda712,null
Armando772,Rold√°n470,null


Encontramos 44 registros de pacientes com alguma parte do seu nome contendo caracteres não convencionais. <p>
Para a ocasião da montagem da tabela de pacientes na camada Silver, iremos substituir tais caracteres por "_", para que futuras apresentações e impressões desses dados em relatórios de analytics não fiquem prejudicadas.

In [0]:
%sql

-- Busca de inconsistências nos campos de data de nascimento e morte

SELECT MAX(DeathDate), MIN(DeathDate), MAX(BirthDate), MIN(BirthDate) FROM bronze.patients

max(DeathDate),min(DeathDate),max(BirthDate),min(BirthDate)
2022-01-27,2011-02-03,1991-11-27,1922-03-24


Como os dados mostram que o paciente mais velho nasceu em 1922 e o mais novo em 1991, concluímos não haver data de nascimento inconsistente.
Em relação à data de morte, tampouco há inconsistência, pois a morte mais antiga foi em 2011 e a mais recente em 2022.
Obs: em etapa anterior, vimos que há casos de NULL em DeathDate, o que indica que são pacientes vivos na época da coleta dos dados.

In [0]:
%sql

--Verificação das categorias existentes para o Gênerio

SELECT DISTINCT Gender FROM bronze.patients

Gender
F
M


In [0]:
%sql

--Verificação das categorias existentes para o estado civil

SELECT DISTINCT Marital FROM bronze.patients

Marital
null
M
S


In [0]:
%sql

-- Verificação de quantos pacientes tem o estado civil nulo na base

SELECT COUNT(*) FROM bronze.patients WHERE Marital IS NULL

count(1)
1


In [0]:
%sql

--Verificação das categorias existentes para a raça

SELECT DISTINCT Race FROM bronze.patients

Race
asian
other
white
black
native
hawaiian


In [0]:
%sql

--Verificação das categorias existentes para a etnia

SELECT DISTINCT Ethnicity FROM bronze.patients

Ethnicity
nonhispanic
hispanic


Não se observam valores inválidos para gênero (M - _Male_, F - _Female_),  Estado civil (M - _Married_, S - _Single_), Raça (_asian, black, hawaiian, native, white, other_) e Etnia (_hispanic, nonhispanic_). Há apenas um registro na base em que o Estado Civil é nulo, o que não é um fato preocupante.

No momento da carga dos dados de pacientes na camada Silver, as colunas que compõem as partes do nome (Fisrt, Last, Maiden) terão os caracteres numéricos removidos e será feita a substituição dos caracteres inválidos por "_".

In [0]:
%sql

-- carga da tabela de pacientes na camada Silver, contendo limpeza dos dados
CREATE TABLE silver.patients
USING DELTA
AS

SELECT 
  id,
  regexp_replace(regexp_replace(First, '[0-9]', ''),  '[^a-zA-Z0-9\\s'']', '_')  AS First,
  regexp_replace(regexp_replace(Last, '[0-9]', ''),   '[^a-zA-Z0-9\\s'']', '_')  AS Last,
  regexp_replace(regexp_replace(Maiden, '[0-9]', ''), '[^a-zA-Z0-9\\s'']', '_')  AS Maiden,
  BIRTHDATE, DEATHDATE, PREFIX, SUFFIX, MARITAL, RACE, ETHNICITY, GENDER, BIRTHPLACE, ADDRESS, CITY, STATE, COUNTY, ZIP, LAT, LON
FROM bronze.patients

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) FROM silver.patients

count(1)
974


Todos os dados dos 974 pacientes foram carregados, após limpeza, na camada Silver.

In [0]:
%sql
SELECT * FROM silver.patients LIMIT 20

id,First,Last,Maiden,BIRTHDATE,DEATHDATE,PREFIX,SUFFIX,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON
5605b66b-e92d-c16c-1b83-b8bf7040d51f,Nikita,Erdman,Leannon,1977-03-19,null,Mrs.,null,M,white,nonhispanic,F,Wakefield Massachusetts US,510 Little Station Unit 69,Quincy,Massachusetts,Norfolk County,02186,42.290937381211286,-70.97550306
6e5ae27c-8038-7988-e2c0-25a103f01bfa,Zane,Hodkiewicz,null,1940-02-19,null,Mr.,null,M,white,nonhispanic,M,Brookline Massachusetts US,747 Conn Throughway,Boston,Massachusetts,Suffolk County,02135,42.308831197562505,-71.0631616
8123d076-0886-9007-e956-d5864aa121a7,Quinn,Marquardt,null,1958-06-04,null,Mr.,null,M,white,nonhispanic,M,Gardner Massachusetts US,816 Okuneva Extension Apt 91,Quincy,Massachusetts,Norfolk County,02170,42.26517684888508,-70.96708508
770518e4-6133-648e-60c9-071eb2f0e2ce,Abel,Smitham,null,1928-12-25,2017-09-29,Mr.,null,M,white,hispanic,M,Randolph Massachusetts US,127 Cole Way Unit 95,Boston,Massachusetts,Suffolk County,02118,42.334303740740594,-71.0668012
f96addf5-81b9-0aab-7855-d208d3d352c5,Edwin,Labadie,null,1928-12-25,2014-02-23,Mr.,null,M,white,hispanic,M,Stow Massachusetts US,976 Ziemann Gateway,Boston,Massachusetts,Suffolk County,02125,42.346771403899275,-71.05881297
8e9650d1-788a-78f9-4a28-d08f7f95354a,Frankie,Oberbrunner,null,1928-12-25,null,Mr.,null,M,white,hispanic,M,Boston Massachusetts US,303 Bechtelar Bypass Suite 26,Boston,Massachusetts,Suffolk County,02467,42.37102647,-71.11810672
183df435-4190-060e-8f8e-bf63c572b266,Eilene,Walsh,Wiegand,1957-11-08,null,Mrs.,null,M,asian,nonhispanic,F,Beijing Beijing Municipality CN,235 Lang Parade,Cambridge,Massachusetts,Middlesex County,02142,42.35892760552785,-71.15622361
720560d4-51da-c38c-ee90-c15935278df1,Lowell,Price,null,1972-06-27,null,Mr.,null,M,white,nonhispanic,M,Lowell Massachusetts US,694 Kuhlman Corner Apt 74,Quincy,Massachusetts,Norfolk County,02170,42.297903944576696,-71.01598304
217851b0-5f47-d376-18b9-0fe4ba77207e,Adrian,Gleason,null,1954-03-06,null,Mr.,null,S,black,hispanic,M,Boston Massachusetts US,808 Gottlieb Wall,Boston,Massachusetts,Suffolk County,02126,42.38408414202098,-71.1006892
ff331e5c-ab16-e218-f39a-63e11de1ed75,Eugene,Abernathy,null,1927-07-10,null,Mr.,null,M,native,hispanic,M,Pembroke Massachusetts US,706 Connelly Track Unit 1,Boston,Massachusetts,Suffolk County,02111,42.358519313454885,-71.07859759


### 🧾 Dimensão Organizações (Hospitais)

In [0]:
%sql
-- Dada a existência de apenas uma organização na base, manteremos seu registro intacto na camada Silver
CREATE TABLE silver.organizations
USING DELTA
AS
SELECT * FROM bronze.organizations


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM silver.organizations

Id,NAME,ADDRESS,CITY,STATE,ZIP,LAT,LON
d78e84ec-30aa-3bba-a33a-f29a3a454662,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,02114,42.362813,-71.069187


### 🧾 Dimensão Pagadores (Seguradoras)

In [0]:
%sql
-- Na camada Silver, manteremos todas as linhas, porém incluiremos apenas colunas que poderão ser de interesse para responder as perguntas do problema e futuras novas consultas
CREATE TABLE silver.payers
USING DELTA
AS
SELECT id, Name, City, State_Headquartered FROM bronze.payers

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM silver.payers 

id,Name,City,State_Headquartered
b3221cfc-24fb-339e-823d-bc4136cbc4ed,Dual Eligible,Baltimore,MD
7caa7254-5050-3b5e-9eae-bd5ea30e809c,Medicare,Baltimore,MD
7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,Medicaid,Baltimore,MD
d47b3510-2895-3b70-9897-342d681c769d,Humana,Louisville,KY
6e2f1a2d-27bd-3701-8d08-dae202c58632,Blue Cross Blue Shield,Chicago,IL
5059a55e-5d6e-34d1-b6cb-d83d16e57bcf,UnitedHealthcare,Minnetonka,MN
4d71f845-a6a9-3c39-b242-14d25ef86a8d,Aetna,Hartford,CT
047f6ec3-6215-35eb-9608-f9dda363a44c,Cigna Health,Bloomfield,CT
42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,Anthem,Indianapolis,IN
b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,NO_INSURANCE,null,null


### 💡 Fato Encounters (Visitas Hospitalares)

Na nossa modelagem em esquema estrela, a tabela **Encounters** da camada Silver é a nossa tabela fato.
<p>
As outras três que criamos, acima, são tabelas dimensão (Patients, Payers, Organizations).

In [0]:
%sql

-- Busca de inconsistências nos campos de data e hora de início e fim de atendimentos

SELECT MAX(Start), MIN(Start), MAX(Stop), MIN(Stop) FROM bronze.encounters

max(Start),min(Start),max(Stop),min(Stop)
2022-02-05T20:27:36Z,2011-01-02T09:26:36Z,2022-02-05T20:42:36Z,2011-01-02T12:58:36Z


Nessa tabela também não encontramos insconsistências de datas e verifica-se que o período dos registros da nossa tabela fato foi de janeiro/2011 a fevereiro/2022.

In [0]:
%sql
-- Busca das categorias da coluna EncounterClass

SELECT count(*) AS Qtd, EncounterClass FROM bronze.encounters GROUP BY EncounterClass ORDER BY Qtd DESC

Qtd,EncounterClass
12537,ambulatory
6300,outpatient
3666,urgentcare
2322,emergency
1931,wellness
1135,inpatient


In [0]:
%sql
-- Busca das categorias da coluna ReasonDescription

SELECT count(*) AS Qtd, ReasonDescription FROM bronze.encounters GROUP BY ReasonDescription ORDER BY Qtd DESC

Qtd,ReasonDescription
19541,null
1738,Chronic congestive heart failure (disorder)
1565,Hyperlipidemia
1341,Normal pregnancy
732,Viral sinusitis (disorder)
723,Malignant neoplasm of breast (disorder)
400,Acute viral pharyngitis (disorder)
352,Acute bronchitis (disorder)
191,Alzheimer's disease (disorder)
115,Sinusitis (disorder)


Pelo que se observa, na base há 6 categorias distintas para _EncounterClass_ e 74 para _ReasonDescription_. Tais colunas serão então mantidas na construção da tabela de Encounters na camada Silver, dado o possível uso desses dados em métricas sobre o dataset.

In [0]:
%sql
-- Criação e carga da tabela fato Encounters na camada Silver
CREATE TABLE silver.encounters
USING DELTA
AS
SELECT id, Patient, Payer, Organization, Start, Stop, EncounterClass, 
Base_Encounter_Cost, Total_Claim_Cost, Payer_Coverage, ReasonDescription
FROM bronze.encounters


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM silver.encounters LIMIT 20

id,Patient,Payer,Organization,Start,Stop,EncounterClass,Base_Encounter_Cost,Total_Claim_Cost,Payer_Coverage,ReasonDescription
32c84703-2481-49cd-d571-3899d5820253,3de74169-7f67-9304-91d4-757e0f3a14d2,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-02T09:26:36Z,2011-01-02T12:58:36Z,ambulatory,85.55,1018.02,0,null
c98059da-320a-c0a6-fced-c8815f3e3f39,d9ec2e44-32e9-9148-179a-1653348cc4e2,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-03T05:44:39Z,2011-01-03T06:01:42Z,outpatient,142.58,2619.36,0,null
4ad28a3a-2479-782b-f29c-d5b3f41a001e,73babadf-5b2b-fee7-189e-6f41ff213e01,7caa7254-5050-3b5e-9eae-bd5ea30e809c,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-03T14:32:11Z,2011-01-03T14:47:11Z,outpatient,85.55,461.59,305.27,null
c3f4da61-e4b4-21d5-587a-fbc89943bc19,3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-03T16:24:45Z,2011-01-03T16:39:45Z,wellness,136.8,1784.24,0,null
a9183b4f-2572-72ea-54c2-b3cd038b4be7,fa006887-d93c-d302-8b89-f3c25f88c0e1,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-03T17:36:53Z,2011-01-03T17:51:53Z,ambulatory,85.55,234.72,0,Hyperlipidemia
c4923a74-3e40-8b0c-cf73-05b9c0390621,823c6b40-9dbe-e463-310b-ea2b23b23b48,7caa7254-5050-3b5e-9eae-bd5ea30e809c,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-03T19:08:16Z,2011-01-03T19:23:16Z,wellness,136.8,1183.25,946.58,null
c140ed81-040e-8319-e860-f72b4738ed22,9c616fc0-00ea-249f-d073-1f3bb15d41fa,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-03T22:39:50Z,2011-01-03T22:54:50Z,outpatient,85.55,6024.77,0,null
2cfd4ddd-ad13-fe1e-528b-15051cea2ec3,d856d6e6-4c98-e7a2-129b-44076c63d008,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-04T14:49:55Z,2011-01-04T15:04:55Z,ambulatory,85.55,11855.19,11205.43,Malignant tumor of colon
16bdc066-886f-34e1-38fa-afb85090b637,a80b1160-93f0-db7e-9f23-04ea6fdddfaf,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-04T15:13:10Z,2011-01-04T15:28:10Z,wellness,136.8,272.8,0,null
17966936-0878-f4db-128b-a43ae10d0878,bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,7caa7254-5050-3b5e-9eae-bd5ea30e809c,d78e84ec-30aa-3bba-a33a-f29a3a454662,2011-01-05T04:02:09Z,2011-01-05T04:17:09Z,outpatient,85.55,9881.17,7872.94,Non-small cell lung cancer (disorder)


## 🥇 3.3. Camada Gold

Tendo já ocorrido a limpeza e a organização na camada Silver dos dados de potencial interesse para Analytics, parte-se para a construção e carga da base de dados na tabela Gold.

Optou-se por um modelo de única tabela _flat_ para os dados na camada Gold. Mais algumas colunas foram desconsideradas, sobretudo na tabela _Encounters_.

Acrescentou-se uma coluna calculada com a duração dos atendimento, a partir das colunas _Start_ e _Stop_. 

In [0]:
%sql
-- Criação da base de dados para representar a camada Silver
CREATE DATABASE IF NOT EXISTS gold;

In [0]:
%sql

-- Na camada Gold, manteremos todas as linhas, porém incluiremos apenas colunas que poderão de interesse para responder as perguntas do problema
-- Acrescentada a nova coluna Duracao_minutos, com o cálculo de timestamp (Stop - Start), em minutos.

CREATE TABLE gold.flat
USING DELTA
AS

SELECT 
  PAT.*, 
  TRY_CAST(E.Start AS TIMESTAMP) AS Inicio,
  TRY_CAST(E.Stop AS TIMESTAMP) AS Final,
  timestampdiff(MINUTE, TRY_CAST(Inicio AS TIMESTAMP), TRY_CAST(Final AS TIMESTAMP)) AS Duracao_minutos,
  PAY.Name AS Pagador,
  PAY.City AS Cidade_Pagador,
  O.Name AS Hospital,
  E.EncounterClass AS EncoutnerClass,
  E.Total_Claim_Cost AS Total_Cost,
  E.Payer_Coverage AS Payer_Coverage,
  E.ReasonDescription AS ReasonDescription

 FROM silver.patients AS PAT 
 LEFT JOIN silver.encounters AS E
    ON PAT.id = E.patient
    LEFT JOIN silver.payers AS PAY
      ON PAY.id = E.Payer
      LEFT JOIN silver.organizations AS O
      ON O.id = E.Organization
 ORDER BY Inicio


num_affected_rows,num_inserted_rows


Observação: optou-se por LEFT JOIN para essa junção, uma vez que a base pode comportar registros de pacientes que ainda não tenham tido nenhum atendimento. Isto é, pacientes registrados em _Patients_, mas sem referência em _Encounters_.

As colunas resultantes na tabela _flat_ da camada Gold são essas mostradas abaixo, em que a semântica dos itens se mantém do dataset original, conforme visto na seção anterior de Catálogo de Dados.



col_name           | data_type
-------------------|---------------
id                 | string
First              | string
Last               | string
Maiden             | string
BIRTHDATE          | string
DEATHDATE          | string
PREFIX             | string
SUFFIX             | string
MARITAL            | string
RACE               | string
ETHNICITY          | string
GENDER             | string
BIRTHPLACE         | string
ADDRESS            | string
CITY               | string
STATE              | string
COUNTY             | string
ZIP                | string
LAT                | string
LON                | string
Inicio             | timestamp
Final              | timestamp
Duracao_minutos    | bigint
Pagador            | string
Cidade_Pagador     | string
Hospital           | string
EncoutnerClass     | string
Total_Cost         | string
Payer_Coverage     | string
ReasonDescription  | string

In [0]:
%sql
SELECT * FROM gold.flat
LIMIT (30)

id,First,Last,Maiden,BIRTHDATE,DEATHDATE,PREFIX,SUFFIX,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,Inicio,Final,Duracao_minutos,Pagador,Cidade_Pagador,Hospital,EncoutnerClass,Total_Cost,Payer_Coverage,ReasonDescription
3de74169-7f67-9304-91d4-757e0f3a14d2,Mariano,O_Kon,null,1928-08-25,2017-02-04,Mr.,null,M,white,nonhispanic,M,Palermo Sicily IT,531 Little Crossing,Boston,Massachusetts,Suffolk County,02132,42.35816095388989,-71.03776628,2011-01-02T09:26:36.000+0000,2011-01-02T12:58:36.000+0000,212,NO_INSURANCE,null,MASSACHUSETTS GENERAL HOSPITAL,ambulatory,1018.02,0,null
d9ec2e44-32e9-9148-179a-1653348cc4e2,Myrtis,Lindgren,Anderson,1964-01-05,2020-06-02,Mrs.,null,M,white,nonhispanic,F,Southborough Massachusetts US,860 Goldner Trailer,Boston,Massachusetts,Suffolk County,02203,42.348269122142355,-71.04225099,2011-01-03T05:44:39.000+0000,2011-01-03T06:01:42.000+0000,17,NO_INSURANCE,null,MASSACHUSETTS GENERAL HOSPITAL,outpatient,2619.36,0,null
73babadf-5b2b-fee7-189e-6f41ff213e01,Marianne,Nienow,White,1924-06-30,null,Mrs.,null,M,asian,nonhispanic,F,Beijing Beijing Municipality CN,760 Klein Rue Apt 38,Boston,Massachusetts,Suffolk County,02131,42.31053436901197,-71.02850623,2011-01-03T14:32:11.000+0000,2011-01-03T14:47:11.000+0000,15,Medicare,Baltimore,MASSACHUSETTS GENERAL HOSPITAL,outpatient,461.59,305.27,null
3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,Efrain,Dibbert,null,1923-05-21,2021-01-04,Mr.,null,M,white,nonhispanic,M,Lowell Massachusetts US,475 Wunsch Overpass,Boston,Massachusetts,Suffolk County,02121,42.36292149492703,-71.01306739,2011-01-03T16:24:45.000+0000,2011-01-03T16:39:45.000+0000,15,NO_INSURANCE,null,MASSACHUSETTS GENERAL HOSPITAL,wellness,1784.24,0,null
fa006887-d93c-d302-8b89-f3c25f88c0e1,Emerson,Kreiger,null,1952-11-02,null,Mr.,null,M,white,nonhispanic,M,Billerica Massachusetts US,586 Torphy Burg,Braintree,Massachusetts,Norfolk County,02184,42.25193860345687,-70.99053529,2011-01-03T17:36:53.000+0000,2011-01-03T17:51:53.000+0000,15,Anthem,Indianapolis,MASSACHUSETTS GENERAL HOSPITAL,ambulatory,234.72,0,Hyperlipidemia
823c6b40-9dbe-e463-310b-ea2b23b23b48,Walter,Prohaska,null,1922-10-09,null,Mr.,null,M,white,nonhispanic,M,Boston Massachusetts US,314 Rowe Key Apt 91,Medford,Massachusetts,Middlesex County,02145,42.42046778751541,-71.12129935,2011-01-03T19:08:16.000+0000,2011-01-03T19:23:16.000+0000,15,Medicare,Baltimore,MASSACHUSETTS GENERAL HOSPITAL,wellness,1183.25,946.58,null
9c616fc0-00ea-249f-d073-1f3bb15d41fa,Dannie,Grimes,Pfannerstill,1985-09-16,null,Mrs.,null,M,white,nonhispanic,F,Andover Massachusetts US,148 Wilkinson Glen Suite 91,Boston,Massachusetts,Suffolk County,02131,42.35808214,-70.9864169,2011-01-03T22:39:50.000+0000,2011-01-03T22:54:50.000+0000,15,NO_INSURANCE,null,MASSACHUSETTS GENERAL HOSPITAL,outpatient,6024.77,0,null
d856d6e6-4c98-e7a2-129b-44076c63d008,Tyree,Hyatt,null,1954-08-26,2012-10-17,Mr.,null,M,white,nonhispanic,M,Shrewsbury Massachusetts US,685 Murphy Terrace Suite 13,Boston,Massachusetts,Suffolk County,02111,42.344093447889726,-71.04088194,2011-01-04T14:49:55.000+0000,2011-01-04T15:04:55.000+0000,15,Medicaid,Baltimore,MASSACHUSETTS GENERAL HOSPITAL,ambulatory,11855.19,11205.43,Malignant tumor of colon
a80b1160-93f0-db7e-9f23-04ea6fdddfaf,Abraham,Ruecker,null,1928-07-03,2014-07-29,Mr.,null,M,white,nonhispanic,M,North Scituate Massachusetts US,725 Hermann Mill Unit 4,Cambridge,Massachusetts,Middlesex County,02472,42.38621815792245,-71.06196589,2011-01-04T15:13:10.000+0000,2011-01-04T15:28:10.000+0000,15,NO_INSURANCE,null,MASSACHUSETTS GENERAL HOSPITAL,wellness,272.8,0,null
bc9d59c3-0a30-6e3b-f47d-022e4f03c8de,Simon,Grant,null,1929-10-14,2013-06-02,Mr.,null,S,white,nonhispanic,M,Lynn Massachusetts US,424 McLaughlin Dale Unit 42,Boston,Massachusetts,Suffolk County,02210,42.336612839127895,-70.98982661,2011-01-05T04:02:09.000+0000,2011-01-05T04:17:09.000+0000,15,Medicare,Baltimore,MASSACHUSETTS GENERAL HOSPITAL,outpatient,9881.17,7872.94,Non-small cell lung cancer 

Na próxima sessão serão feitas consultas na camada Gold a fim de responder às questões propostas nesse trabalho.

<br/><br/>

> Siga para o próximo notebook [🚀](https://github.com/cristianofanchin/puc-rio/blob/main/engenhariadados/4-Solucao_do_Problema_e_Autoavaliacao.ipynb)

> [◀](https://github.com/cristianofanchin/puc-rio/blob/main/engenhariadados/README.md) Volte para o início 